# Initialize

In [ ]:
import pandas as pd
from Bio import SeqIO
import sys
import os
from ast import literal_eval
import numpy as np

#For plotting
from matplotlib.colors import LinearSegmentedColormap
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

#For statistics
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.stats import pearsonr

import re
from Bio import SeqIO
import ast # for safe eveal, for parsing some of the data


import const #to reload use import(importlib) and then importlib.reload(const)
from const import MPRA_data_paths
from const import pos_active_ctrl_color,neg_active_ctrl_color,highlight_color,custom_cmap
from const import set_equal_plot_limits
from const import plot_color_pallete
from const import custom_cmap_bolder

const.set_plot_style()

#to reload consts without restarting kernel
#import importlib
#importlib.reload(const)

In [ ]:
# Use CPM normalization?
cpm=True

# Use log scale in visualization?
logScale=False

#Additional filters?
min_DNA_reads = 5 

### Import

In [ ]:
##Import oligos
library = "humanMPRA_L3a2" #humanMPRA_L3a2 humanMPRA_L1a1 humanMPRA_L1a1_Neurons #modern_humanMPRA_NPC Max_MPRA_run2 PMID_38766054_Reilly thylacine_biorxiv_Gallego_Romero
library_paths = MPRA_data_paths[library]

In [ ]:
#output directory
output_path = "/home/labs/davidgo/Collaboration/MPRA_QC_pipeline/activity/output/"+library+'/'

if "comb_df" in library_paths:
    print('loading activity_df...')
    activity_df = pd.read_csv(library_paths["comb_df"])
    
if "activity_per_rep" in library_paths:
    print('loading activity_per_rep...')
    activity_by_rep_df = pd.read_csv(library_paths["activity_per_rep"])

if "UMI_counts" in library_paths:
    print('loading UMI_counts...')
    UMI_counts = pd.read_csv(library_paths["UMI_counts"],sep = '\t')
    #UMI_counts.rename(columns={'oligo_bc.1': 'oligo diversity'}, inplace=True)
    #UMI_counts['oligo diversity'] = 1/UMI_counts['oligo diversity']

if "oligo_fasta" in library_paths:
    print('loading oligo_fasta...')
    fasta_file = library_paths["oligo_fasta"]

print('Done!')
                                                 
#hMPRA specific data
positive_ctrls_corr_dir = "/home/labs/davidgo/Collaboration/humanMPRA/additional/analyze_controls/chondrocytes/controls_df.csv"
positive_ctrls_corr_df = pd.read_csv(positive_ctrls_corr_dir)

### Create dictionary of postive controls, negative controls, highlighted oligos

In [ ]:
if library == 'Max_MPRA_run2':
    oligo_groups = ['POSITIVE_CONTROL','NEGATIVE_CONTROL','SLEA'] # NM 26-7-25 I have no idea what SLEA is, I just put it as a filler for the highlight oligos
else: 
    oligo_groups = ['NegCtrl_not_active','PosCtrl_osteoblast','PosCtrl_diff']

groups_dict = {}

for group in oligo_groups:
    compiled_pattern =  re.compile(group)
    group_oligos = set()
    
    with open(fasta_file, "r") as infile:
        for record in SeqIO.parse(infile, "fasta"):
            if compiled_pattern.search(record.id):
                group_oligos.add(record.id)
    groups_dict[group] = group_oligos

    
if library == 'Max_MPRA_run2':
    pos_olg = groups_dict['POSITIVE_CONTROL']  
    neg_olg = groups_dict['NEGATIVE_CONTROL']
    highlight_olg = groups_dict['SLEA']
elif library == 'PMID_38766054_Reilly':
    # Path to your Excel file
    supplementary_path = library_paths['supplementary']
    ctrl_oligo_seqs = pd.read_excel(supplementary_path, sheet_name="S23.MPRA Controls", skiprows=1)
    pos_olg = ctrl_oligo_seqs[ctrl_oligo_seqs["Control Type"] == "ctrl:pos"]["ID"].dropna().astype(str).tolist()
    neg_olg = ctrl_oligo_seqs[ctrl_oligo_seqs["Control Type"] == "ctrl:neg"]["ID"].dropna().astype(str).tolist()  
else: 
    pos_olg = groups_dict['PosCtrl_osteoblast']  
    neg_olg = groups_dict['NegCtrl_not_active']
    highlight_olg = groups_dict['PosCtrl_diff']

In [ ]:
# Add normalization
DNA_sum = activity_df["DNA_rep_comb"].sum()
RNA_sum = activity_df["RNA_rep_comb"].sum()

activity_df["DNA_rep_comb_cpm"] = 1000000*(activity_df["DNA_rep_comb"]+1)/DNA_sum
activity_df["RNA_rep_comb_cpm"] = 1000000*(activity_df["RNA_rep_comb"]+1)/RNA_sum

activity_df["DNA_rep_comb_cpm_log"] = np.log2(activity_df["DNA_rep_comb_cpm"])
activity_df["RNA_rep_comb_cpm_log"] = np.log2(activity_df["RNA_rep_comb_cpm"])


activity_df = activity_df[activity_df["DNA_rep_comb"] >= min_DNA_reads]

DNA_counts = "DNA_rep_comb"
RNA_counts = "RNA_rep_comb"

if cpm:
    DNA_counts=DNA_counts+"_cpm"
    RNA_counts=RNA_counts+"_cpm"

if logScale:
    DNA_counts=DNA_counts+"_log"
    RNA_counts=RNA_counts+"_log"

# Evaluating DNA and RNA complexity

### Retained cCREs and barcodes

In [ ]:
#print("Number of cCREs after association step:", oligo_count_after_association)

In [ ]:
activity_by_rep_df_vectorized = activity_by_rep_df[['RNA_filtered_std2_rep1','DNA_filtered_std2_rep1',
                                                   'RNA_filtered_std2_rep2','DNA_filtered_std2_rep2',
                                                   'RNA_filtered_std2_rep3','DNA_filtered_std2_rep3']]
def safe_eval(x):
    if pd.isna(x):   # catches np.nan, pd.NA, None
        return np.nan
    try:
        return ast.literal_eval(str(x))
    except (ValueError, SyntaxError):
        return x  # fallback: return as-is if it cannot be parsed

activity_by_rep_df_vectorized = activity_by_rep_df_vectorized.applymap(safe_eval)


In [ ]:
results = {}

for col in activity_by_rep_df_vectorized.columns:
    print(col)
    series = activity_by_rep_df_vectorized[col]

    # drop NAs and ensure all entries are lists
    series = series.dropna().apply(lambda x: np.asarray(x, dtype=float))

    # 1. Fraction of rows with at least one non-zero
    frac_rows_nonzero = (series.apply(lambda arr: np.any(arr != 0))).mean()

    # 2. Flatten everything into one list and compute fraction non-zero
    if len(series) > 0:
        flat = np.concatenate([np.atleast_1d(x) for x in series.values])
        frac_values_nonzero = np.count_nonzero(flat) / flat.size
    else:
        frac_values_nonzero = np.nan

    results[col] = {
        "cCRE": frac_rows_nonzero,
        "Barcode": frac_values_nonzero,
    }

results_df = pd.DataFrame(results).T

# reset index so column names become a column
results_df = results_df.reset_index().rename(columns={"index": "rep"})

# Extract measurement type (before "_rep")
results_df["measurement"] = results_df["rep"].str.split("_rep").str[0]


# Melt into tidy format
results_melted = results_df.melt(
    id_vars=["rep", "measurement"],
    value_vars=["cCRE", "Barcode"],
    var_name="metric",
    value_name="fraction"
)

print(results_melted)


In [ ]:
print('NOTE: need to fix the x axis in illustrator')
# Clean measurement name
results_melted["measurement_clean"] = results_melted["measurement"].str.replace("_filtered_std2", "")

# Build combined label with replicate number
results_melted["group_rep"] = (
    results_melted["metric"] + " " +
    results_melted["measurement_clean"] + " " +
    results_melted["rep"].str.extract(r'(rep\d+)')[0]
)

# Desired order (12 bars total, clustered by metric)
order = [
    "cCRE DNA rep1",
    "cCRE DNA rep2",
    "cCRE DNA rep3",
    "cCRE RNA rep1",
    "cCRE RNA rep2",
    "cCRE RNA rep3",
    "Barcode DNA rep1",
    "Barcode DNA rep2",
    "Barcode DNA rep3",
    "Barcode RNA rep1",
    "Barcode RNA rep2",
    "Barcode RNA rep3"
]

# Assign colors: map each group to x or y
palette = {grp: (plot_color_pallete['cCRE'] if "cCRE" in grp else plot_color_pallete['barcode']) for grp in order}

plt.figure(figsize=(12,6))
sns.barplot(
    data=results_melted,
    x="group_rep",
    y="fraction",
    order=order,
    palette=palette,
    ci=None
)

plt.ylabel("Fraction retained")
plt.xlabel("")
plt.yticks([0,1])

plt.xticks(rotation=45, ha="right")
plt.tight_layout()

const.save_fig(plt,'retained_cCREs_and_barcodes',output_path)

plt.show()


### Reads per UMI

In [ ]:
plt.clf()

plt.hist(data=UMI_counts,x='oligo diversity', color='grey',bins = 55)
plt.xlim(1,5)
#plt.xticks(np.arange(1, 6, 1))
plt.xticks([1,5])

yticks = plt.gca().get_yticks()
plt.yticks([yticks[0], yticks[-1]])

#plt.yscale('log')
plt.xlabel('Reads per UMI')
plt.ylabel('Number of barcodes')


const.save_fig(plt,'UMI_complexity_histogram',output_path)

plt.show()

### Activity distribution

In [ ]:
plt.clf()

bin_edges = np.linspace(-4, 4, 201)  # 100 bins between -10 and 20

# First histogram (all scores)
plt.hist(
    data=activity_df,
    x='ratio_log_rep_comb', bins=bin_edges, color=plot_color_pallete['default_color'], label='Non-active cCREs')

# Second histogram (filtered scores)
plt.hist(
    data=activity_df[activity_df['activity_adjusted'] == 'active'],
    x='ratio_log_rep_comb', bins=bin_edges, color='red', label='Active cCREs')

plt.xlabel(r'$\log_{2}\!\left(\frac{\mathrm{RNA}}{\mathrm{DNA}}\right)$')
plt.ylabel('Number of cCREs')
plt.legend()
stat,pval=stats.skewtest(activity_df['ratio_log_rep_comb'].dropna())
#plt.text(x=plt.xlim()[0],y=plt.ylim()[1], s=f"Skewness statistic: {stat:.2f}\n p-value: {pval:.2f}")
plt.text(
    0.05, 0.95,  # relative position in axes coordinates
    f"Skew test:\nstat = {stat:.2f}\np = {pval:.3g}",
    ha='left', va='top',
    transform=plt.gca().transAxes,
    fontsize=10,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.xticks([-4,0,4])

yticks = plt.gca().get_yticks()
plt.yticks([yticks[0], yticks[-1]])

plt.legend(
    loc="upper right",
    fontsize=10,         # smaller text
    markerscale=0.8,    # shrink the color boxes
    handlelength=1,     # shorter legend lines
    handletextpad=0.5,  # less padding between box and text
    borderpad=0.3       # tighter box
)


const.save_fig(plt,'activity_distribution',output_path)
plt.show()




### P-value distribution

In [ ]:
# Drop NAs and sort p-values
pvals = activity_df['pval.mad'].dropna().sort_values().to_numpy()
n = len(pvals)

# Expected quantiles under Uniform(0,1)
expected = np.linspace(0, 1, n, endpoint=False) + 0.5/n   # mid-point adjustment

plt.figure(figsize=(6,6))
plt.scatter(expected, pvals, s=10, color='black', alpha=0.6,linewidths=0.1)
plt.plot([0,1], [0,1], color='black', linestyle='--')  # y=x line

plt.xticks([0,1])
plt.yticks([0,1])


plt.xlabel("Expected p-values\n (Uniform(0,1))")
plt.ylabel("Observed p-values")
plt.tight_layout()

const.save_fig(plt,'P_value_distribution',output_path)

plt.show()


### Downsampling analysis - active cCREs

### Cumulative RNA reads

In [ ]:
activity_df['rna_counts_percentile']=activity_df['RNA_rep_comb'].rank(ascending=False)/activity_df.shape[0]
sorted_activity_df=activity_df.sort_values(by='rna_counts_percentile')
sorted_activity_df['rna_cumulative_sum']=sorted_activity_df['RNA_rep_comb'].cumsum()
sorted_activity_df['rna_cumulative_percentile']=sorted_activity_df['rna_cumulative_sum']/sorted_activity_df['RNA_rep_comb'].sum()

In [ ]:
fig, ax = plt.subplots()
ax.plot(sorted_activity_df['rna_counts_percentile'],
        sorted_activity_df['rna_cumulative_percentile'],
        color = plot_color_pallete['read'],
       linewidth=5)
ax.set_xlabel('cCRE quantile')
ax.set_ylabel('RNA reads cumulative quantile')

plt.xticks([0,1])
plt.yticks([0,1])

const.save_fig(plt,"cumulative_RNA_reads",output_path)

plt.show()

### DNA counts vs GC content (Bookdown only)

In [ ]:
#Processing
# Initialize empty lists to store modified identifiers and sequences

identifiers = []
sequences = []
gc_contents = []

# Parse the FASTA file
for record in SeqIO.parse(fasta_file, "fasta"):
    identifier = record.id  # Remove the first character
    identifiers.append(identifier)
    sequence = str(record.seq)
    sequences.append(sequence)
    
    # Calculate GC content
    gc_count = sequence.count('g') + sequence.count('c')+sequence.count('G') + sequence.count('C')
    total_bases = len(sequence)
    gc_content = (gc_count / total_bases) * 100
    gc_contents.append(gc_content)

# Create a DataFrame
gc_df = pd.DataFrame({'oligo': identifiers, 'Sequence': sequences, 'GC_Content': gc_contents})


In [ ]:
# merge this to activity_df
merged_gc_activity = activity_df.merge(gc_df, on = "oligo", how = "left")
merged_gc_activity.loc[:,'DNA_rep_comb_log10'] = np.log10(merged_gc_activity['DNA_rep_comb'])
merged_gc_activity.loc[:,'DNA_rep_comb_clipped_500'] = merged_gc_activity['DNA_rep_comb'].clip(upper=500, inplace=False)
merged_gc_activity.loc[:, 'GC_Content_clipped_25_60'] = merged_gc_activity['GC_Content'].clip(lower=25, upper=60, inplace=False)

In [ ]:
# take oligos with specifc DNA count and compare high vs low gc
bins = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]
labels = [
    '0-5', '5-10', '10-15', '15-20', '20-25', '25-30', '30-35', '35-40',
    '40-45', '45-50', '50-55', '55-60', '60-65', '65-70', '70-75', '75-80',
    '80-85', '85-90', '90-95', '95-100'
]


merged_gc_activity.loc[:,'GC_Content_label'] = pd.cut(merged_gc_activity['GC_Content'], bins=bins, labels=labels, right=True)


In [ ]:

f, (ax_DNA ,ax_hist) = plt.subplots(2, sharex=True,gridspec_kw={"height_ratios": (.5,.5)})
sns.boxplot(
    x=merged_gc_activity["GC_Content_label"],
    y=merged_gc_activity["DNA_rep_comb"],
    showfliers=False,
    color=plot_color_pallete['read'],
    ax=ax_DNA)

# Set y-axis limits and ticks for DNA boxplot
ax_DNA.set_ylim(bottom=0)
yticks_DNA = ax_DNA.get_yticks()
ax_DNA.set_yticks([yticks_DNA[0], yticks_DNA[-1]])

sns.histplot(
    x=merged_gc_activity["GC_Content_label"],
    color=plot_color_pallete['cCRE'],
    ax=ax_hist)

# Set x-axis limits to show 20-80% range
# Find the positions of categories within the filtered data
categories = [label for label in labels if label in merged_gc_activity["GC_Content_label"].cat.categories]
first_cat_pos = 0
last_cat_pos = len(categories) - 1

ax_hist.set_xlim(-0.5, last_cat_pos + 0.5)
ax_DNA.set_xlim(-0.5, last_cat_pos + 0.5)

# Set x-axis ticks to show only "0" and "100"
ax_hist.set_xticks([first_cat_pos, last_cat_pos])
ax_hist.set_xticklabels(['0', '100'], rotation=45)

ax_DNA.set_ylabel("DNA reads")
ax_hist.set_xlabel("%GC")
ax_hist.set_ylabel("cCREs")

# Set y-axis limits and ticks for histogram
ax_hist.set_ylim(bottom=0)
yticks_hist = ax_hist.get_yticks()
ax_hist.set_yticks([yticks_hist[0], yticks_hist[-1]])

const.save_fig(plt,'GC_content_bias_in_DNA',output_path)
plt.show()

# Evaluating reproducability

### Similarity between samples (PCA) - Waiting for David

### Correlation between replicates

In [ ]:
# Prepare the data
# Replace Inf values with NaN, then drop any rows with NaN values
activity_by_rep_df = activity_by_rep_df.replace([np.inf, -np.inf], np.nan)

# Drop rows where either 'ratio_log_filtered_std2_rep1' or 'ratio_log_filtered_std2_rep2' has NaN or Inf
activity_by_rep_df = activity_by_rep_df.dropna(subset=['ratio_log_filtered_std2_rep1', 'ratio_log_filtered_std2_rep2'])


x = activity_by_rep_df['ratio_log_filtered_std2_rep1'].values
y = activity_by_rep_df['ratio_log_filtered_std2_rep2'].values


# Create the KDE (Kernel Density Estimate)
values = np.vstack([x, y])
kernel = gaussian_kde(values)

# Evaluate the KDE for each data point
density = kernel(values)

In [ ]:
max_density_threshold = 0.1 

# Clip values in density before coloring
density_capped = np.clip(density, a_min=None, a_max=max_density_threshold)

plt.clf()

scatter = plt.scatter(
    x, y,
    c=density_capped,      # use capped values
    cmap=custom_cmap_bolder,
    s=10, edgecolors='none'
)

plt.xlabel(r'$\log_{2}\!\left(\frac{\mathrm{RNA}}{\mathrm{DNA}}\right)$ replicate 1')
plt.ylabel(r'$\log_{2}\!\left(\frac{\mathrm{RNA}}{\mathrm{DNA}}\right)$ replicate 2')

plt.xlim(-4, 6)
plt.ylim(-4, 6)
plt.xticks([-3,0,3,6])
plt.yticks([-3,0,3,6])  

const.save_fig(plt, 'RNA_DNA_ratio_correlation_between_replicates', output_path)
plt.show()


### Variation at various activity levels

### Correlation between replicates (controls)

In [ ]:
plt.clf()

scatter = plt.scatter(
    x, y, c='lightgray', s=10, edgecolors='none')

plt.xlabel(r'$\log_{2}\!\left(\frac{\mathrm{RNA}}{\mathrm{DNA}}\right)$ replicate 1')
plt.ylabel(r'$\log_{2}\!\left(\frac{\mathrm{RNA}}{\mathrm{DNA}}\right)$ replicate 2')

plt.scatter(data=activity_by_rep_df[activity_by_rep_df["oligo"].isin(neg_olg)],
                x = 'ratio_log_filtered_std2_rep1',
                y = 'ratio_log_filtered_std2_rep2',
                s=15,label = 'negative control', color = neg_active_ctrl_color)

plt.scatter(data=activity_by_rep_df[activity_by_rep_df["oligo"].isin(pos_olg)],
                x = 'ratio_log_filtered_std2_rep1',
                y = 'ratio_log_filtered_std2_rep2',
                s=15,label = 'positive control', color = pos_active_ctrl_color)


set_equal_plot_limits(x,y)

plt.xlim(-4, 6)
plt.ylim(-4, 6)
plt.xticks([-3,0,3,6])
plt.yticks([-3,0,3,6])  

plt.legend()

const.save_fig(plt,'RNA_DNA_ratio_correlation_between_replicates_with_controls',output_path)
plt.show()

## Activity of controls - sample comparison (always hMPRA chonds)

In [ ]:
#Requiers a specific dataset of control activity in various batches
positive_ctrls_corr_dir = "/home/labs/davidgo/Collaboration/humanMPRA/additional/analyze_controls/chondrocytes/controls_df.csv"
positive_ctrls_corr_df = pd.read_csv(positive_ctrls_corr_dir)

alpha = positive_ctrls_corr_df.filter(regex='alpha')

corr_alpha = alpha.corr()
plt.clf()
sns.heatmap(corr_alpha, cmap="Blues", annot=True)


const.save_fig(plt,'positive_controls_corr_between_libs_hMPRA',output_path)
plt.show()


## Outlier barcodes + min(DNA)

## Outlier barcodes

# Evaluating dynamic range

## RNA vs DNA

## Activity of Controls

In [ ]:
palette = [
    pos_active_ctrl_color,  # 'PosCtrl_osteoblast_active'
    pos_active_ctrl_color,  # 'PosCtrl_neuron_active'
    pos_active_ctrl_color,  # 'PosCtrl_chondrocyte_active'
    pos_active_ctrl_color,  # 'PosCtrl_diff'
    pos_active_ctrl_color,  # 'NegCtrl_active_not_diff'
    neg_active_ctrl_color,  # 'NegCtrl_not_active'
    neg_active_ctrl_color,  # 'scrambled_control'
    neg_active_ctrl_color,  # 'NegCtrl_non_SCREEN'
    highlight_color         # 'No control'
]


if re.findall("a4",library):
    print(library)
    activity_df.loc[activity_df['oligo'].str.contains('posCtrlActive_BJ'), 'control_type'] = 'posCtrlActive_BJ'
    activity_df.loc[activity_df['oligo'].str.contains('posCtrlActive_adipocytes'), 'control_type'] = 'posCtrlActive_adipocytes'
    activity_df.loc[activity_df['oligo'].str.contains('posCtrlActive_NPC'), 'control_type'] = 'posCtrlActive_NPC'
    activity_df.loc[activity_df['oligo'].str.contains('posCtrlActive_HOb'), 'control_type'] = 'posCtrlActive_HOb'
    activity_df.loc[activity_df['oligo'].str.contains('NegCtrl_not_active'), 'control_type'] = 'NegCtrl_not_active'
    activity_df.loc[activity_df['oligo'].str.contains('scrambled_control'), 'control_type'] = 'scrambled_control'
    activity_df.loc[activity_df['oligo'].str.contains('PosCtrl_diff'), 'control_type'] = 'PosCtrl_diff'
    activity_df['control_type'] = activity_df['control_type'].fillna(value='No control')

    sns.boxplot(data=activity_df[~activity_df['control_type'].isin(('PosCtrl_diff','NegCtrl_active_not_diff'))],
            x="mad.score", y="control_type", showfliers = False,linewidth= 2,
            order=['posCtrlActive_BJ',
                   'posCtrlActive_HOb',
                   'posCtrlActive_adipocytes',
                   'posCtrlActive_NPC',
                   'NegCtrl_not_active',
                   'scrambled_control',
                   'No control'],
            palette=[pos_active_ctrl_color,pos_active_ctrl_color,pos_active_ctrl_color,pos_active_ctrl_color,neg_active_ctrl_color,neg_active_ctrl_color,highlight_color])
elif library == 'archaic_MPRA':
    print(library)
    activity_df.loc[activity_df['oligo'].str.contains('posCtrlActive_BJ76'), 'control_type'] = 'PosCtrl_BJ'
    activity_df.loc[activity_df['oligo'].str.contains('NegCtrl_not_active'), 'control_type'] = 'NegCtrl_not_active'
    activity_df.loc[activity_df['oligo'].str.contains('scrambled_control'), 'control_type'] = 'scrambled_control'
    activity_df['control_type'] = activity_df['control_type'].fillna(value='No control')
    
    sns.boxplot(data=activity_df[~activity_df['control_type'].isin(('PosCtrl_diff','NegCtrl_active_not_diff'))],
            x="mad.score", y="control_type", showfliers = False,linewidth= 2,
            order=['PosCtrl_adipocyte_active',
                   'PosCtrl_neuron_active',
                   'PosCtrl_fibroblast_active',
                   'PosCtrl_osteoblast_active',
                   'NegCtrl_not_active',
                   'scrambled_control',
                  'No control'],
            palette=[pos_active_ctrl_color,pos_active_ctrl_color,pos_active_ctrl_color,pos_active_ctrl_color,neg_active_ctrl_color,neg_active_ctrl_color,highlight_color])

elif library == 'Max_MPRA_run2':
    print(library)
    sns.boxplot(data=activity_df,
            x="mad.score", y="control_type", showfliers = False,linewidth= 2,
            order=['PosCtrl',
                   'NegCtrl',
                  'No control'],
            palette=[pos_active_ctrl_color,neg_active_ctrl_color,highlight_color])
    
elif library == 'PMID_38766054_Reilly':
    print(library)
    conditions = [
        activity_df["oligo"].isin(pos_olg),
        activity_df["oligo"].isin(neg_olg)
    ]
    choices = ["Positive control", "Negative control"]
    activity_df["control_type"] = np.select(conditions, choices, default="cCREs")

    sns.boxplot(data=activity_df,
            x="statistic", y="control_type", showfliers = False,linewidth= 2,
            order=['Positive control',
                   'Negative control',
                  'cCREs'],
            palette=[pos_active_ctrl_color,neg_active_ctrl_color,highlight_color])
else:  
    activity_df.loc[activity_df['oligo'].str.contains('PosCtrl_osteoblast_active'), 'control_type'] = 'PosCtrl_osteoblast_active'
    activity_df.loc[activity_df['oligo'].str.contains('PosCtrl_chondrocyte_active'), 'control_type'] = 'PosCtrl_chondrocyte_active'
    activity_df.loc[activity_df['oligo'].str.contains('PosCtrl_neuron_active'), 'control_type'] = 'PosCtrl_neuron_active'
    activity_df.loc[activity_df['oligo'].str.contains('NegCtrl_non_SCREEN'), 'control_type'] = 'NegCtrl_non_SCREEN'
    activity_df.loc[activity_df['oligo'].str.contains('NegCtrl_active_not_diff'), 'control_type'] = 'NegCtrl_active_not_diff'
    activity_df.loc[activity_df['oligo'].str.contains('scrambled_control'), 'control_type'] = 'scrambled_control'
    activity_df.loc[activity_df['oligo'].str.contains('NegCtrl_not_active'), 'control_type'] = 'NegCtrl_not_active'
    activity_df.loc[activity_df['oligo'].str.contains('PosCtrl_diff'), 'control_type'] = 'PosCtrl_diff'
    activity_df['control_type'] = activity_df['control_type'].fillna(value='No control')
    
    sns.boxplot(data=activity_df[filtered_data[~activity_df['control_type'].isin(('PosCtrl_diff','NegCtrl_active_not_diff'))]],
                x="mad.score", y="control_type", showfliers = False,linewidth= 2,
                order=['PosCtrl_osteoblast_active','PosCtrl_neuron_active','PosCtrl_chondrocyte_active','NegCtrl_not_active','scrambled_control','No control'],
                palette=[pos_active_ctrl_color,pos_active_ctrl_color,pos_active_ctrl_color,neg_active_ctrl_color,neg_active_ctrl_color,highlight_color])

plt.xlabel("Activity")
plt.ylabel("Control type")

xticks = plt.gca().get_xticks()
plt.xticks([xticks[0],0, xticks[-1]])

const.save_fig(plt,'Control_activity_boxplots',output_path)

plt.show()


## Genomic annotations

## Proximity to TSS

 ## Correlation between AI predictions and MPRA results